In [ ]:
pip install datasets numpy pandas scipy tqdm

In [ ]:
import re
import random
import numpy as np
import pandas as pd

from datasets import load_dataset
from collections import Counter, defaultdict
from scipy.sparse import csr_matrix
from tqdm import tqdm

In [ ]:

LANGUAGES = [
    "asm_Beng",
    "ben_Beng",
    "brx_Deva",
    "doi_Deva",
    "gom_Deva",
    "guj_Gujr",
    "hin_Deva",
    "kan_Knda",
    "kas_Arab",
    "mai_Deva",
    "mal_Mlym",
    "mar_Deva",
    "mni_Mtei",
    "npi_Deva",
    "ory_Orya",
    "pan_Guru",
    "san_Deva",
    "snd_Deva",
    "tam_Taml",
    "tel_Telu",
    "urd_Arab",
    "khasi",
    "santhali"
]

In [ ]:
def sentence_tokenize(text):

    if not isinstance(text, str):
        return []

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Split on:
    # ।  Devanagari danda
    # ॥ Double danda
    # . English period
    # ! Exclamation
    # ? Question mark
    sentences = re.split(r'(?<=[.!?।॥])\s+', text)

    sentences = [
        s.strip()
        for s in sentences
        if len(s.strip()) >= 10
    ]

    return sentences

In [ ]:
def collect_sentences(language, target=1000):

    print(f"Collecting {language}...")

    dataset = load_dataset(
        "ai4bharat/IndicCorpV2",
        "indiccorp_v2",
        split=language,
        streaming=True
    )

    sentences = []

    for row in dataset:

        text = row["text"]

        # Normalize whitespace
        text = re.sub(r"\s+", " ", text).strip()

        # Sentence splitting
        parts = re.split(
            r"(?<=[.!?।॥])\s+",
            text
        )

        for sentence in parts:

            sentence = sentence.strip()

            # Remove very short / very long sentences
            if 10 <= len(sentence) <= 500:
                sentences.append(sentence)

            if len(sentences) >= target:
                break

        if len(sentences) >= target:
            break

    return sentences[:target]

In [ ]:
sentences = collect_sentences("hin_Deva", 10)

print(len(sentences))

for i, sentence in enumerate(sentences):
    print(i + 1, sentence)

README.md:   0%|          | 0.00/4.01k [00:00<?, ?B/s]

10
1 लोगों को बिलों संबंधी सुविधा देना ही उनका काम
2 इनेलो 1987 में उस वक्त ऐसे ही दोराहे पर खड़ी थी, जब पूर्व उपप्रधानमंत्री देवीलाल ने अपने पुत्र ओमप्रकाश चौटाला को अपना राजनीतिक उत्तराधिकारी घोषित किया था।
3 हालांकि तब पार्टी पर देवीलाल की मजबूत पकड़ के चलते पार्टी टूटने से बच गई थी।
4 1989 में देवीलाल केन्द्र की राजनीति में सक्रिय हो गए थे और उनके उपप्रधानमंत्री बनने के पश्चात् उनके तीन बेटों जगदीश सिंह, रणजीत सिंह और ओमप्रकाश चौटाला में से रणजीत और ओमप्रकाश के बीच हरियाणा में उनकी राजनीतिक विरासत को लेकर जंग शुरू हो गई थी।
5 उन परिस्थितियों में देवीलाल ने कड़ा निर्णय लेते हुए पार्टी की बागडोर ओमप्रकाश चौटाला के हवाले कर दी थी, जिसके बाद रणजीत की बगावत का असर पार्टी, संगठन और उनकी सरकार पर भी पड़ा था।
6 उस समय रणजीत की नाराजगी के चलते उनके समर्थन में कई कैबिनेट मंत्रियों ने इस्तीफे दे दिए थे किन्तु तब पार्टी सुप्रीमो चौ.
7 देवीलाल की हरियाणा की जनता पर इतनी मजबूत पकड़ थी कि ओमप्रकाश चौटाला को उत्तराधिकारी बनाने के उनके फैसले का जनता के बीच कोई खास विरोध नहीं हुआ था लेकिन आज स्थिति 

In [ ]:
all_data = []

for language in LANGUAGES:

    sentences = collect_sentences(
        language,
        target=5000
    )

    print(
        f"{language}: {len(sentences)} sentences"
    )

    for sentence in sentences:

        all_data.append({
            "text": sentence,
            "label": language
        })


df = pd.DataFrame(all_data)

print("\nDataset shape:", df.shape)

print("\nSamples per language:")
print(df["label"].value_counts())

asm_Beng: 5000 sentences
ben_Beng: 5000 sentences
brx_Deva: 5000 sentences
doi_Deva: 5000 sentences
gom_Deva: 5000 sentences
guj_Gujr: 5000 sentences
hin_Deva: 5000 sentences
kan_Knda: 5000 sentences
kas_Arab: 3220 sentences
mai_Deva: 5000 sentences
mal_Mlym: 5000 sentences
mar_Deva: 5000 sentences
mni_Mtei: 5000 sentences
npi_Deva: 5000 sentences
ory_Orya: 5000 sentences
pan_Guru: 5000 sentences
san_Deva: 5000 sentences
snd_Deva: 5000 sentences
tam_Taml: 5000 sentences
tel_Telu: 5000 sentences
urd_Arab: 5000 sentences
khasi: 5000 sentences
santhali: 5000 sentences

Dataset shape: (113220, 2)

Samples per language:
label
asm_Beng    5000
ben_Beng    5000
brx_Deva    5000
doi_Deva    5000
gom_Deva    5000
guj_Gujr    5000
hin_Deva    5000
kan_Knda    5000
mai_Deva    5000
mal_Mlym    5000
mar_Deva    5000
npi_Deva    5000
mni_Mtei    5000
ory_Orya    5000
pan_Guru    5000
tel_Telu    5000
san_Deva    5000
snd_Deva    5000
tam_Taml    5000
khasi       5000
urd_Arab    5000
santhali    50

In [ ]:
df.to_csv(
    "indic_language_dataset.csv",
    index=False,
    encoding="utf-8"
)

In [ ]:
df = pd.read_csv(
    "indic_language_dataset.csv"
)

In [ ]:
print(df.info())

print("Dataset shape:", df.shape)
print(df.sample(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113220 entries, 0 to 113219
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    113220 non-null  object
 1   label   113220 non-null  object
dtypes: object(2)
memory usage: 1.7+ MB
None
Dataset shape: (113220, 2)
                                                     text     label
104893  The Congress on Monday announced a massive peo...     khasi
108512                                    ᱚᱱ-ᱫᱷᱚᱱ ᱟᱨᱡᱟᱣᱟᱭ  santhali
84672   اقتدار جي راند سان سلهاڙيل ڪا مزيدار ڪهاڻي هلي...  snd_Deva
108526  ᱥᱤᱞᱪᱚᱨ ᱨᱮᱫᱚ ᱥᱤᱢᱟᱹᱛ ᱤᱱᱰᱚᱥᱴᱨᱤ ᱜᱮ ᱢᱮᱱᱟᱜᱼᱟ ᱟᱨ ᱵᱟᱹᱲ...  santhali
86158                                          ٿيسز ناٽڪ.  snd_Deva


In [ ]:
import numpy as np

np.random.seed(42)

train_parts = []
val_parts = []
test_parts = []

for language in LANGUAGES:

    language_df = df[
        df["label"] == language
    ].sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    train_parts.append(
        language_df.iloc[:800]
    )

    val_parts.append(
        language_df.iloc[800:900]
    )

    test_parts.append(
        language_df.iloc[900:1000]
    )

train_df = pd.concat(
    train_parts,
    ignore_index=True
)

val_df = pd.concat(
    val_parts,
    ignore_index=True
)

test_df = pd.concat(
    test_parts,
    ignore_index=True
)

In [ ]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (18400, 2)
Validation: (2300, 2)
Test: (2300, 2)


In [ ]:
print("TRAIN")
print(train_df["label"].value_counts().sort_index())

print("\nVALIDATION")
print(val_df["label"].value_counts().sort_index())

print("\nTEST")
print(test_df["label"].value_counts().sort_index())

TRAIN
label
asm_Beng    800
ben_Beng    800
brx_Deva    800
doi_Deva    800
gom_Deva    800
guj_Gujr    800
hin_Deva    800
kan_Knda    800
kas_Arab    800
khasi       800
mai_Deva    800
mal_Mlym    800
mar_Deva    800
mni_Mtei    800
npi_Deva    800
ory_Orya    800
pan_Guru    800
san_Deva    800
santhali    800
snd_Deva    800
tam_Taml    800
tel_Telu    800
urd_Arab    800
Name: count, dtype: int64

VALIDATION
label
asm_Beng    100
ben_Beng    100
brx_Deva    100
doi_Deva    100
gom_Deva    100
guj_Gujr    100
hin_Deva    100
kan_Knda    100
kas_Arab    100
khasi       100
mai_Deva    100
mal_Mlym    100
mar_Deva    100
mni_Mtei    100
npi_Deva    100
ory_Orya    100
pan_Guru    100
san_Deva    100
santhali    100
snd_Deva    100
tam_Taml    100
tel_Telu    100
urd_Arab    100
Name: count, dtype: int64

TEST
label
asm_Beng    100
ben_Beng    100
brx_Deva    100
doi_Deva    100
gom_Deva    100
guj_Gujr    100
hin_Deva    100
kan_Knda    100
kas_Arab    100
khasi       100
mai_Deva  

In [ ]:
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
label_to_id = {
    label: i
    for i, label in enumerate(LANGUAGES)
}

id_to_label = {
    i: label
    for label, i in label_to_id.items()
}

In [ ]:
y_train = np.array([
    label_to_id[label]
    for label in train_df["label"]
])

y_val = np.array([
    label_to_id[label]
    for label in val_df["label"]
])

y_test = np.array([
    label_to_id[label]
    for label in test_df["label"]
])

In [ ]:
import re

def normalize_text(text):

    text = str(text).lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [ ]:
def get_word_ngrams(text):

    text = normalize_text(text)

    words = text.split()

    features = []

    # Word unigram
    for word in words:
        features.append(
            ("WORD1", word)
        )

    # Word bigram
    for i in range(len(words) - 1):

        bigram = (
            words[i],
            words[i + 1]
        )

        features.append(
            ("WORD2", bigram)
        )

    return features

In [ ]:
def get_char_ngrams(text):

    text = normalize_text(text)

    features = []

    for n in [2, 3, 4]:

        for i in range(
            len(text) - n + 1
        ):

            gram = text[i:i+n]

            features.append(
                (f"CHAR{n}", gram)
            )

    return features

In [ ]:
def extract_features(text):

    features = []

    features.extend(
        get_word_ngrams(text)
    )

    features.extend(
        get_char_ngrams(text)
    )

    return features

In [ ]:
sample = train_df.iloc[0]["text"]

features = extract_features(sample)

print(sample)
print()
print(features[:30])

দুৰ্গা পূজাৰ আনন্দৰ মাজতে আজি অসমত পুনৰ দুগৰাকী কোভিড আক্ৰান্তৰ মৃত্যু হয়।

[('WORD1', 'দুৰ্গা'), ('WORD1', 'পূজাৰ'), ('WORD1', 'আনন্দৰ'), ('WORD1', 'মাজতে'), ('WORD1', 'আজি'), ('WORD1', 'অসমত'), ('WORD1', 'পুনৰ'), ('WORD1', 'দুগৰাকী'), ('WORD1', 'কোভিড'), ('WORD1', 'আক্ৰান্তৰ'), ('WORD1', 'মৃত্যু'), ('WORD1', 'হয়।'), ('WORD2', ('দুৰ্গা', 'পূজাৰ')), ('WORD2', ('পূজাৰ', 'আনন্দৰ')), ('WORD2', ('আনন্দৰ', 'মাজতে')), ('WORD2', ('মাজতে', 'আজি')), ('WORD2', ('আজি', 'অসমত')), ('WORD2', ('অসমত', 'পুনৰ')), ('WORD2', ('পুনৰ', 'দুগৰাকী')), ('WORD2', ('দুগৰাকী', 'কোভিড')), ('WORD2', ('কোভিড', 'আক্ৰান্তৰ')), ('WORD2', ('আক্ৰান্তৰ', 'মৃত্যু')), ('WORD2', ('মৃত্যু', 'হয়।')), ('CHAR2', 'দু'), ('CHAR2', 'ুৰ'), ('CHAR2', 'ৰ্'), ('CHAR2', '্গ'), ('CHAR2', 'গা'), ('CHAR2', 'া '), ('CHAR2', ' প')]


<h3><B>TF-IDF

LOG(N+1/n+1)+1

In [ ]:
from collections import Counter
from scipy.sparse import csr_matrix   #sparse matrix
from tqdm import tqdm
import numpy as np


class CustomTfidf:

    def __init__(self):

        self.vocabulary = {}
        self.idf = None

    def fit(self, texts):

        document_frequency = Counter()

        print("Building vocabulary...")

        for text in tqdm(texts):

            features = extract_features(text)

            # A feature counts only once per document
            unique_features = set(features)

            for feature in unique_features:

                document_frequency[feature] += 1

        # Assign integer index
        for index, feature in enumerate(
            document_frequency.keys()
        ):

            self.vocabulary[feature] = index

        N = len(texts)

        self.idf = np.zeros(
            len(self.vocabulary)
        )

        for feature, index in self.vocabulary.items():

            df_value = document_frequency[feature]

            self.idf[index] = (
                np.log(
                    (N + 1) /
                    (df_value + 1)
                )
                + 1
            )

        print(
            "Vocabulary size:",
            len(self.vocabulary)
        )

        return self

    def transform(self, texts):

        rows = []
        columns = []
        values = []

        for row_id, text in enumerate(
            tqdm(texts)
        ):

            features = extract_features(text)

            counts = Counter(features)

            total_terms = sum(
                counts.values()
            )

            for feature, count in counts.items():

                if feature not in self.vocabulary:
                    continue

                column_id = self.vocabulary[
                    feature
                ]

                # TF
                tf = count / total_terms

                # TF-IDF
                value = (
                    tf *
                    self.idf[column_id]
                )

                rows.append(row_id)
                columns.append(column_id)
                values.append(value)

        matrix = csr_matrix(
            (
                values,
                (rows, columns)
            ),
            shape=(
                len(texts),
                len(self.vocabulary)
            )
        )

        return matrix

    def fit_transform(self, texts):

        self.fit(texts)

        return self.transform(texts)

In [ ]:
vectorizer = CustomTfidf()

X_train = vectorizer.fit_transform(
    train_df["text"].tolist()
)

Building vocabulary...


100%|██████████| 18400/18400 [00:09<00:00, 1901.97it/s]


Vocabulary size: 927305


100%|██████████| 18400/18400 [00:11<00:00, 1646.32it/s]


In [ ]:
X_val = vectorizer.transform(
    val_df["text"].tolist()
)

X_test = vectorizer.transform(
    test_df["text"].tolist()
)

100%|██████████| 2300/2300 [00:01<00:00, 1950.26it/s]


In [ ]:
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (18400, 927305)
X_val: (2300, 927305)
X_test: (2300, 927305)


In [ ]:
import numpy as np


class CustomLogisticRegression:

    def __init__(
        self,
        learning_rate=0.5,
        epochs=20,
        batch_size=256,
        reg_strength=1e-4,
        random_state=42
    ):

        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.reg_strength = reg_strength
        self.random_state = random_state

        self.W = None
        self.b = None

    def softmax(self, scores):

        # Numerical stability
        scores = scores - np.max(
            scores,
            axis=1,
            keepdims=True
        )

        exp_scores = np.exp(scores)

        probabilities = (
            exp_scores /
            np.sum(
                exp_scores,
                axis=1,
                keepdims=True
            )
        )

        return probabilities

    def fit(
        self,
        X,
        y,
        X_val=None,
        y_val=None
    ):

        np.random.seed(self.random_state)

        n_samples = X.shape[0]
        n_features = X.shape[1]

        n_classes = len(
            np.unique(y)
        )

        # Weight matrix
        self.W = np.zeros(
            (n_features, n_classes),
            dtype=np.float64
        )

        # Bias
        self.b = np.zeros(
            n_classes,
            dtype=np.float64
        )

        for epoch in range(self.epochs):

            # Shuffle training samples
            indices = np.random.permutation(
                n_samples
            )

            total_loss = 0
            batches = 0

            for start in range(
                0,
                n_samples,
                self.batch_size
            ):

                end = min(
                    start + self.batch_size,
                    n_samples
                )

                batch_indices = indices[
                    start:end
                ]

                X_batch = X[
                    batch_indices
                ]

                y_batch = y[
                    batch_indices
                ]

                # -------------------------
                # Forward pass
                # -------------------------
                #z = XW + b

                scores = (
                    X_batch @ self.W
                    + self.b
                )

                probabilities = self.softmax(
                    scores
                )

                # -------------------------
                # One-hot labels
                # -------------------------

                Y = np.zeros_like(
                    probabilities
                )

                Y[
                    np.arange(
                        len(y_batch)
                    ),
                    y_batch
                ] = 1

                # -------------------------
                # Cross entropy
                # -------------------------

                loss = -np.mean(
                    np.sum(
                        Y *
                        np.log(
                            probabilities + 1e-12
                        ),
                        axis=1
                    )
                )

                # L2 regularization
                loss += (
                    self.reg_strength / 2
                ) * np.sum(
                    self.W ** 2
                )

                total_loss += loss
                batches += 1

                # -------------------------
                # Gradients
                # -------------------------

                error = (
                    probabilities - Y
                )

                grad_W = (
                    X_batch.T @ error
                ) / len(y_batch)

                grad_W += (
                    self.reg_strength *
                    self.W
                )

                grad_b = np.mean(
                    error,
                    axis=0
                )

                # -------------------------
                # Gradient descent
                # -------------------------

                self.W -= (
                    self.learning_rate *
                    grad_W
                )

                self.b -= (
                    self.learning_rate *
                    grad_b
                )

            avg_loss = (
                total_loss / batches
            )

            # Training accuracy
            train_pred = self.predict(X)

            train_accuracy = np.mean(
                train_pred == y
            )

            if X_val is not None:

                val_pred = self.predict(
                    X_val
                )

                val_accuracy = np.mean(
                    val_pred == y_val
                )

                print(
                    f"Epoch {epoch + 1:02d}/{self.epochs} | "
                    f"Loss: {avg_loss:.4f} | "
                    f"Train Acc: {train_accuracy:.4f} | "
                    f"Val Acc: {val_accuracy:.4f}"
                )

            else:

                print(
                    f"Epoch {epoch + 1:02d}/{self.epochs} | "
                    f"Loss: {avg_loss:.4f} | "
                    f"Train Acc: {train_accuracy:.4f}"
                )

    def predict_proba(self, X):

        scores = (
            X @ self.W
            + self.b
        )

        return self.softmax(
            scores
        )

    def predict(self, X):

        probabilities = (
            self.predict_proba(X)
        )

        return np.argmax(
            probabilities,
            axis=1
        )

In [ ]:
model = CustomLogisticRegression(
    learning_rate=0.5,
    epochs=10,
    batch_size=256,
    reg_strength=1e-4
)

In [ ]:
model.fit(
    X_train,
    y_train,
    X_val,
    y_val
)

Epoch 01/10 | Loss: 3.1260 | Train Acc: 0.5955 | Val Acc: 0.5813
Epoch 02/10 | Loss: 3.1044 | Train Acc: 0.6742 | Val Acc: 0.6600
Epoch 03/10 | Loss: 3.0832 | Train Acc: 0.7884 | Val Acc: 0.7739
Epoch 04/10 | Loss: 3.0622 | Train Acc: 0.9346 | Val Acc: 0.9230
Epoch 05/10 | Loss: 3.0414 | Train Acc: 0.9204 | Val Acc: 0.9043
Epoch 06/10 | Loss: 3.0208 | Train Acc: 0.9494 | Val Acc: 0.9317
Epoch 07/10 | Loss: 3.0004 | Train Acc: 0.9107 | Val Acc: 0.8974
Epoch 08/10 | Loss: 2.9802 | Train Acc: 0.9019 | Val Acc: 0.8909
Epoch 09/10 | Loss: 2.9603 | Train Acc: 0.9513 | Val Acc: 0.9304
Epoch 10/10 | Loss: 2.9406 | Train Acc: 0.9418 | Val Acc: 0.9239


In [ ]:
def macro_f1(
    y_true,
    y_pred,
    num_classes
):

    f1_scores = []

    for class_id in range(
        num_classes
    ):

        tp = np.sum(
            (y_true == class_id) &
            (y_pred == class_id)
        )

        fp = np.sum(
            (y_true != class_id) &
            (y_pred == class_id)
        )

        fn = np.sum(
            (y_true == class_id) &
            (y_pred != class_id)
        )

        precision = (
            tp / (tp + fp)
            if (tp + fp) > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (tp + fn) > 0
            else 0.0
        )

        if precision + recall == 0:

            f1 = 0.0

        else:

            f1 = (
                2 *
                precision *
                recall /
                (precision + recall)
            )

        f1_scores.append(f1)

    return np.mean(f1_scores)

In [ ]:
val_predictions = model.predict(
    X_val
)

val_accuracy = np.mean(
    val_predictions == y_val
)

val_f1 = macro_f1(
    y_val,
    val_predictions,
    len(LANGUAGES)
)

print(
    "Validation Accuracy:",
    val_accuracy
)

print(
    "Validation Macro-F1:",
    val_f1
)

Validation Accuracy: 0.9239130434782609
Validation Macro-F1: 0.9212545806192051


In [ ]:
test_predictions = model.predict(
    X_test
)

test_accuracy = np.mean(
    test_predictions == y_test
)

test_f1 = macro_f1(
    y_test,
    test_predictions,
    len(LANGUAGES)
)

print(
    "Test Accuracy:",
    test_accuracy
)

print(
    "Test Macro-F1:",
    test_f1
)

Test Accuracy: 0.918695652173913
Test Macro-F1: 0.9152159267551572


In [ ]:
def confusion_matrix_custom(
    y_true,
    y_pred,
    num_classes
):

    matrix = np.zeros(
        (num_classes, num_classes),
        dtype=int
    )

    for true, pred in zip(
        y_true,
        y_pred
    ):

        matrix[true, pred] += 1

    return matrix


cm = confusion_matrix_custom(
    y_test,
    test_predictions,
    len(LANGUAGES)
)

print(cm)


pd.DataFrame(
    cm,
    index=LANGUAGES,
    columns=LANGUAGES
).to_csv(
    "confusion_matrix.csv"
)

[[ 99   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   1   0]
 [  4  96   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0]
 [  0   0  83   0   0   0  16   0   0   0   0   0   0   0   0   0   1   0
    0   0   0   0   0]
 [  0   0   2  85   0   0  12   0   0   0   0   0   0   0   0   0   1   0
    0   0   0   0   0]
 [  0   0   7   0  32   0   4   0   0   0   0  48   0   0   0   0   2   0
    0   0   0   7   0]
 [  0   0   0   0   0 100   0   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0]
 [  0   0   2   0   0   0  98   0   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0]
 [  0   0   0   0   0   0   0 100   0   0   0   0   0   0   0   0   0   0
    0   0   0   0   0]
 [  0   0   0   0   0   0   0   0  97   0   0   0   0   0   0   0   0   0
    0   0   3   0   0]
 [  0   0  14   0   0   0  14   0   0  65   0   0   0   1   0   0   6   0
    0   0   0   0   0]
 [  0   0   0   0   0   0   0 

In [ ]:
def identify_language(text):

    X = vectorizer.transform(
        [text]
    )

    probabilities = (
        model.predict_proba(X)[0]
    )

    predicted_id = np.argmax(
        probabilities
    )

    predicted_language = (
        id_to_label[predicted_id]
    )

    confidence = (
        probabilities[predicted_id]
    )

    return (
        predicted_language,
        confidence
    )



text = "भारत एक बहुत विविध देश है।"

language, confidence = identify_language(
    text
)

print("Language:", language)
print("Confidence:", confidence)

100%|██████████| 1/1 [00:00<00:00, 1630.76it/s]

Language: hin_Deva
Confidence: 0.04785544076032596
